# NLA — Step 1: Harvest activations

This notebook runs a frozen small language model over text and saves its internal activations.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

Outputs are written to your Google Drive so a disconnect never loses work.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the project code

Replace `REPO_URL` with your GitHub repo (keep it public during development for the simplest setup; you can make it private and share with `monperrus` at the end).

In [ ]:
REPO_URL = "https://github.com/mohamedibrahim26/nla-kth.git"

import os
repo_dir = "/content/" + REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.exists(repo_dir):
    !git clone $REPO_URL
%cd $repo_dir
!pip install -q -r requirements.txt

## 3. Mount Google Drive (for saving outputs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/nla/data'
os.makedirs(DATA_DIR, exist_ok=True)
print('Saving activations to:', DATA_DIR)

## 4. Harvest activations

Start small (e.g. 500) to confirm everything works, then re-run with the full count.

In [ ]:
!python src/harvest_activations.py --num_samples 500 --data_dir "$DATA_DIR"

## 5. Peek at what we collected

In [ ]:
import numpy as np, json
acts = np.load(os.path.join(DATA_DIR, 'activations.npy'))
print('activations shape:', acts.shape)
norms = np.linalg.norm(acts.astype(np.float32), axis=1)
print('L2 norm mean/std: %.2f / %.2f' % (norms.mean(), norms.std()))
with open(os.path.join(DATA_DIR, 'metadata.jsonl')) as f:
    first = json.loads(f.readline())
print('\nExample snippet the model read:')
print(repr(first['text'][:300]))